# 数据准备

02 从原始 CSV 独立开始，不依赖 01 运行时生成的变量。01 已核对标签、`-` 分布和完全重复行，并指出整行去重后仍需检查模型输入是否重复。

为了确认使用既定数据版本，先核对路径、文件哈希和列名顺序，再删除完全重复行。后续整理输入并检查普通分层划分的重叠情况。普通划分用于核对处理过程，不交给 03 训练模型。后续模型比较使用输入组合隔离划分，具体条件在建立该划分时说明。


In [2]:
# 1.核对原始文件并删除完全重复行
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd

root = Path.cwd().parent if Path.cwd().name == "experiment_steps" else Path.cwd()
data_path = root / "data" / "raw" / "Train_Test_Network.csv"
assert data_path.is_file(), f"找不到数据文件：{data_path}"
expected_sha256 = "65d5465df1809b984fd10e4703bc9c012d1aefd11803773856364216f0a3520d"
expected_columns = [
    "ts", "src_ip", "src_port", "dst_ip", "dst_port", "proto", "service",
    "duration", "src_bytes", "dst_bytes", "conn_state", "missed_bytes",
    "src_pkts", "src_ip_bytes", "dst_pkts", "dst_ip_bytes", "dns_query",
    "dns_qclass", "dns_qtype", "dns_rcode", "dns_AA", "dns_RD", "dns_RA",
    "dns_rejected", "ssl_version", "ssl_cipher", "ssl_resumed",
    "ssl_established", "ssl_subject", "ssl_issuer", "http_trans_depth",
    "http_method", "http_uri", "http_version", "http_request_body_len",
    "http_response_body_len", "http_status_code", "http_user_agent",
    "http_orig_mime_types", "http_resp_mime_types", "weird_name",
    "weird_addl", "weird_notice", "label", "type",
]
digest = hashlib.sha256()
with data_path.open("rb") as source:
    for chunk in iter(lambda: source.read(1024 * 1024), b""):
        digest.update(chunk)
raw_sha256 = digest.hexdigest()
print("原始文件：", data_path.resolve())
print("SHA-256：", raw_sha256)
assert raw_sha256 == expected_sha256, "原始 CSV 与已核对版本不同"

raw = pd.read_csv(data_path, low_memory=False)
assert raw.columns.tolist() == expected_columns, "原始列名或列顺序发生变化"
df = raw.drop_duplicates().copy()
print("原始行列数：", raw.shape)
print("完全重复行数：", raw.duplicated().sum())
print("去重后行列数：", df.shape)
display(df["label"].value_counts().sort_index())

原始文件： /home/wyh/projects/cybersecurity_learning/data/raw/Train_Test_Network.csv
SHA-256： 65d5465df1809b984fd10e4703bc9c012d1aefd11803773856364216f0a3520d
原始行列数： (461043, 45)
完全重复行数： 11071
去重后行列数： (449972, 45)


label
0    288929
1    161043
Name: count, dtype: int64

结论：当前文件通过既定哈希和列名顺序检查。原始数据为 461,043 行，其中 11,071 行完全重复，去重后剩 449,972 行。原始文件没有被改写。

`label` 直接区分正常与攻击，`type` 表示正常或具体攻击类型。两列包含待预测的答案，所以不能进入模型输入。`label` 保留为训练目标，`type` 保留用于类别统计和后续漏报分析。

时间戳和 IP 可能记录本次采集的时间与主机身份，端口也包含服务和通信信息。为了评估不使用这五列的检测方案，下一步排除 `ts`、`src_ip`、`dst_ip`、`src_port`、`dst_port`，并核对剩余列名和数量。这是当前实验的输入范围选择，不表示五列没有预测价值，也不能据此断言对其他环境更有利。


In [3]:
# 2.排除时间、IP、端口和标签并核对输入列
identifier_cols = ["ts", "src_ip", "src_port", "dst_ip", "dst_port"]
target_cols = ["label", "type"]
X = df.drop(columns=identifier_cols + target_cols).copy()
y = df["label"].copy()
print("输入列数：", X.shape[1])
print("目标类别：", y.value_counts().sort_index().to_dict())
print("输入列名：", X.columns.tolist())
assert X.shape[1] == 38
assert not set(identifier_cols + target_cols) & set(X.columns)

输入列数： 38
目标类别： {0: 288929, 1: 161043}
输入列名： ['proto', 'service', 'duration', 'src_bytes', 'dst_bytes', 'conn_state', 'missed_bytes', 'src_pkts', 'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes', 'dns_query', 'dns_qclass', 'dns_qtype', 'dns_rcode', 'dns_AA', 'dns_RD', 'dns_RA', 'dns_rejected', 'ssl_version', 'ssl_cipher', 'ssl_resumed', 'ssl_established', 'ssl_subject', 'ssl_issuer', 'http_trans_depth', 'http_method', 'http_uri', 'http_version', 'http_request_body_len', 'http_response_body_len', 'http_status_code', 'http_user_agent', 'http_orig_mime_types', 'http_resp_mime_types', 'weird_name', 'weird_addl', 'weird_notice']


结论：排除五列时间、IP、端口信息及两列标签后，保留 38 列模型输入。

01 已查看各列的数据类型、不同取值数和 `-` 分布。`-` 出现在 `service` 以及 DNS、SSL、HTTP、weird 等记录中。[数据来源说明](https://research.unsw.edu.au/projects/toniot-datasets)指出，网络数据采用 Zeek（Bro）工具记录。[Zeek 文本日志约定](https://docs.zeek.org/en/v8.2.1/scripts/base/frameworks/logging/writers/ascii.zeek.html#id-LogAscii::unset_field)默认用 `-` 表示未赋值的可选条目。

结合这一约定，本次将相关列中的 `-` 按未提供取值处理，统一记为 `n/a`。这样可保留其与实际取值的区别，不把它当成数值零。具体未提供取值的原因不能仅从当前 CSV 判断，仍需追溯原始日志和采集配置。

各列的信息含义不同，所以不采用同一种转换。连接时长、字节数、数据包数及 HTTP 消息体长度表示测量值，保留其数值。[DNS 日志定义](https://docs.zeek.org/en/v8.2.0/scripts/base/protocols/dns/main.zeek.html)与 [HTTP 日志定义](https://docs.zeek.org/en/v8.2.0/scripts/base/protocols/http/main.zeek.html)说明了查询类别、查询类型、响应代码和状态码的含义。对应的 `dns_qclass`、`dns_qtype`、`dns_rcode`、`http_status_code` 是代码或编号，不能把大小直接解释为测量强弱。为了先固定基础输入并控制类别展开的列数，这四列暂按现有数值保留。该选择尚未与类别表示作对照，不视为最优处理。

协议、连接状态和标志等取值表示不同类别。为了保留类别差异，并避免编号引入人为的大小顺序，采用独热编码。01 已核对 `weird_addl` 的 `43`、`46`、`48` 及 `weird_notice` 的 `F`，基础方案将它们作为类别保留。类别集合在划分后由训练集确定。

为了先建立输入数量较少的基础方案，查询、证书、URI、用户代理、MIME 类型及异常名称等十列只记录是否提供取值。0 表示未提供，1 表示提供。这些列原本并非二值列，二值化会舍弃具体文本，以及版本、加密套件和异常名称等差异。它是否足够用于检测，还需与保留具体取值的方案比较。

下一步按上述方案设置数值、二值和类别三组，核对列名是否覆盖全部输入，再处理 `-` 与二值取值。实际列数在运行后汇总。


In [4]:
# 3.整理`-`标记并确定各列的编码方式
binary_cols = [
    "dns_query", "ssl_version", "ssl_cipher", "ssl_subject", "ssl_issuer",
    "http_uri", "http_user_agent", "http_orig_mime_types",
    "http_resp_mime_types", "weird_name",
]
category_cols = [
    "proto", "service", "conn_state", "dns_AA", "dns_RD", "dns_RA",
    "dns_rejected", "ssl_resumed", "ssl_established", "http_trans_depth",
    "http_method", "http_version", "weird_addl", "weird_notice",
]
number_cols = [col for col in X.columns if col not in binary_cols + category_cols]
assert len(number_cols) + len(binary_cols) + len(category_cols) == 38
assert all(pd.api.types.is_numeric_dtype(X[col]) for col in number_cols)

X[binary_cols + category_cols] = X[binary_cols + category_cols].replace("-", "n/a")
for col in binary_cols:
    X[col] = X[col].ne("n/a").astype("int8")
assert not X.isna().any().any()
print("数值列、二值列、类别列：", len(number_cols), len(binary_cols), len(category_cols))
print("service 的 n/a 数量：", X["service"].eq("n/a").sum())
print("weird_notice 的取值：", X["weird_notice"].value_counts().to_dict())

数值列、二值列、类别列： 14 10 14
service 的 n/a 数量： 269985
weird_notice 的取值： {'n/a': 448678, 'F': 1294}


结论：38 列按既定方案分为 14 个保留数值的列、10 个已转换为二值表示的列和 14 个保留类别取值的列。相关列中的 `-` 已替换为 `n/a`。十个二值列用 0 表示未提供取值，1 表示提供取值；这个含义仅适用于这十列。数值列中的 0 应按原始数据项解释。类别列保留 `n/a` 这一取值，`weird_notice` 也保留在类别组。

去重后的正常与攻击数量不同。为了减少随机划分造成的二分类比例偏差，下一步先按 `label` 分层，建立 80% 训练集、20% 测试集的普通划分。这样保留较多训练记录，也留出测试记录。比例是预先固定的安排，不代表最优比例已经验证。这里先核对普通划分，不把它作为最终模型输入。

随机种子固定为 42，便于重复得到相同划分。42 没有更优的含义。按 `label` 分层不能保证九种攻击分别保持相同比例，所以同时展示全部 `type` 的数量。


In [5]:
# 4.按正常与攻击比例建立 80 比 20 的普通分层划分
from sklearn.model_selection import train_test_split
random_seed = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=random_seed
)
type_train = df.loc[X_train.index, "type"]
type_test = df.loc[X_test.index, "type"]
print("训练行数：", len(X_train))
print("测试行数：", len(X_test))
print("训练攻击比例：", y_train.mean())
print("测试攻击比例：", y_test.mean())
display(pd.concat([
    type_train.value_counts().rename("训练集"),
    type_test.value_counts().rename("测试集"),
], axis=1).fillna(0).astype(int))
assert not set(X_train.index) & set(X_test.index)

训练行数： 359977
测试行数： 89995
训练攻击比例： 0.35789508774171686
测试攻击比例： 0.3578976609811656


,训练集,测试集
type,,
normal,231143,57786
scanning,16073,3927
dos,16071,3929
ddos,16054,3946
password,16021,3979
ransomware,15991,4009
backdoor,15955,4045
xss,15934,4066
injection,15890,4110


结论：普通分层划分得到 359,977 条训练记录和 89,995 条测试记录，两部分攻击比例接近。上表展示全部流量类型的数量。`type` 用于统计和结果解释，不进入模型输入。

01 已指出，删除完全重复行后，排除时间、IP 和端口仍可能使不同记录具有相同输入。行索引没有交集，不能证明输入组合互不重复。

所以下一步用每行处理后的 38 列取值生成指纹，筛查跨训练集和测试集的相同输入候选。指纹可能碰撞，候选仍需逐列核对。


In [6]:
# 5.用输入指纹筛查跨训练集和测试集的重复候选
train_hash = pd.util.hash_pandas_object(X_train, index=False)
test_hash = pd.util.hash_pandas_object(X_test, index=False)
hash_overlap = int(test_hash.isin(set(train_hash)).sum())
print("测试集中与训练集共享输入指纹的行数：", hash_overlap)
print("测试集总行数：", len(X_test))

测试集中与训练集共享输入指纹的行数： 54061
测试集总行数： 89995


结论：按处理后的 38 列输入计算，有 54,061 条测试记录的指纹在训练集出现。指纹相同只是筛查线索，还不能证明逐列取值一致，也不能说明标签一致。

为了区分输入重复、标签冲突与哈希碰撞，下一步逐列比较这些候选，并分别统计。


In [7]:
# 6.逐列核对重复输入和标签冲突
from collections import Counter

all_hash = pd.util.hash_pandas_object(X, index=False)
hash_sizes = all_hash.value_counts(sort=False)
repeated_hashes = hash_sizes.index[hash_sizes.gt(1)]
audit_index = all_hash.index[all_hash.isin(repeated_hashes)]
audit_values = X.loc[audit_index].to_numpy(dtype=object)
audit_meta = pd.DataFrame({
    "hash": all_hash.loc[audit_index].to_numpy(),
    "label": y.loc[audit_index].to_numpy(),
    "type": df.loc[audit_index, "type"].to_numpy(),
    "in_train": audit_index.isin(X_train.index),
    "position": np.arange(len(audit_index)),
})

shared_test_rows = 0
shared_exact_groups = 0
shared_test_types = Counter()
train_conflict_groups = 0
train_conflict_rows = 0
train_conflict_types = Counter()
test_opposite_rows = 0
test_opposite_types = Counter()
test_new_label_rows = 0
test_new_label_types = Counter()
collision_hash_groups = 0

for fingerprint, block in audit_meta.groupby("hash", sort=False):
    exact_rows = {}
    for position, label, traffic_type, in_train in block[
        ["position", "label", "type", "in_train"]
    ].itertuples(index=False, name=None):
        row_values = tuple(audit_values[position])
        counts = exact_rows.setdefault(row_values, {
            "train_labels": Counter(), "test_labels": Counter(),
            "train_types": Counter(), "test_types": Counter(),
            "test_type_labels": Counter(),
        })
        if in_train:
            counts["train_labels"][int(label)] += 1
            counts["train_types"][traffic_type] += 1
        else:
            counts["test_labels"][int(label)] += 1
            counts["test_types"][traffic_type] += 1
            counts["test_type_labels"][(traffic_type, int(label))] += 1

    if len(exact_rows) > 1:
        collision_hash_groups += 1
    for counts in exact_rows.values():
        train_labels = counts["train_labels"]
        test_labels = counts["test_labels"]
        if train_labels and test_labels:
            shared_exact_groups += 1
            shared_test_rows += sum(test_labels.values())
            shared_test_types.update(counts["test_types"])
            for (traffic_type, label), count in counts["test_type_labels"].items():
                if any(train_label != label for train_label in train_labels):
                    test_opposite_rows += count
                    test_opposite_types[traffic_type] += count
                if label not in train_labels:
                    test_new_label_rows += count
                    test_new_label_types[traffic_type] += count
        if len(train_labels) > 1:
            train_conflict_groups += 1
            train_conflict_rows += sum(train_labels.values())
            train_conflict_types.update(counts["train_types"])
print("逐列确认的共享测试行数：", shared_test_rows)
print("逐列确认的共享输入组合数：", shared_exact_groups)
print("训练侧同时含两类标签的输入组合数：", train_conflict_groups)
print("这些冲突组合中的训练行数：", train_conflict_rows)
print("测试侧可找到相反训练标签的行数：", test_opposite_rows)
print("测试标签在同输入训练侧从未出现的行数：", test_new_label_rows)
print("同指纹但输入不同的哈希组数：", collision_hash_groups)
display(pd.DataFrame({
    "共享测试行": pd.Series(shared_test_types),
    "训练侧冲突组中的行": pd.Series(train_conflict_types),
    "测试侧有相反训练标签的行": pd.Series(test_opposite_types),
    "测试侧训练未见标签的行": pd.Series(test_new_label_types),
}).fillna(0).astype(int))
assert shared_test_rows <= hash_overlap
del audit_values, audit_meta

逐列确认的共享测试行数： 54061
逐列确认的共享输入组合数： 4659
训练侧同时含两类标签的输入组合数： 124
这些冲突组合中的训练行数： 29697
测试侧可找到相反训练标签的行数： 7444
测试标签在同输入训练侧从未出现的行数： 28
同指纹但输入不同的哈希组数： 0


,共享测试行,训练侧冲突组中的行,测试侧有相反训练标签的行,测试侧训练未见标签的行
backdoor,3783,75,17,0
ddos,1809,321,56,0
dos,2834,7,1,0
injection,53,60,10,0
mitm,3,2,1,0
normal,35039,8707,2157,22
password,1143,1960,547,4
ransomware,3999,9733,2395,2
scanning,3062,3,2,0
xss,2336,8829,2258,0


结论：逐列核对确认，54,061 条测试记录共享 4,659 个训练集输入组合。同指纹但输入不同的哈希组数为 0。

训练集有 124 个输入组合同时含正常与攻击标签，涉及 29,697 条训练记录。测试集有 7,444 行能在同输入的训练记录中找到相反标签，其中 28 行的标签在对应训练组合中完全没有出现。前一个数量包括两种标签都已出现的情况，两个数字不是同一统计口径。

相同输入对应不同标签，说明当前输入不足以唯一确定标签，不能直接认定标注错误。普通划分的输入重叠也不能作为独立测试的依据，所以后面需要建立输入组合隔离划分。

为了将类别转换为数值，并避免类别编号产生人为顺序，下一步采用独热编码。训练集中的每一种类别生成一列，1 表示属于该类别，0 表示未标记该类别。测试集使用相同列名和顺序。未见过的类别不会新增列，该原始列的独热部分全部为 0。

数值列保持原值，每列仍对应一个输入列，所以本步骤无需统计取值种数来确定输出列数。已处理的二值列保持 0/1 格式，不再拆列。为了让每个生成的列都能追溯到来源，展示全部原始列名、表示方式、输出列名和数量，不预设最终维数。表中的“训练集类别数”用于核对类别转换，不是训练样本数。


In [15]:
# 7.根据普通分层划分训练集进行独热编码并核对每一列
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
def make_encoder():
    return ColumnTransformer(
        transformers=[
            ("number", "passthrough", number_cols),
            ("binary", "passthrough", binary_cols),
            ("category", OneHotEncoder(handle_unknown="ignore", sparse_output=False,
                                       dtype=np.float64), category_cols),
        ],
        remainder="drop",
        sparse_threshold=0,
        verbose_feature_names_out=False,
    )
encoder = make_encoder()
train_encoded = encoder.fit_transform(X_train).astype(np.float64, copy=False)
test_encoded = encoder.transform(X_test).astype(np.float64, copy=False)
feature_names = encoder.get_feature_names_out()

feature_rows = []
for field in number_cols:
    feature_rows.append((field, "数值", "无需统计", 1, field))
for field in binary_cols:
    feature_rows.append((field, "二值", int(X_train[field].nunique()), 1, field))
onehot = encoder.named_transformers_["category"]
onehot_names = onehot.get_feature_names_out(category_cols).tolist()
cursor = 0
for field, categories in zip(category_cols, onehot.categories_):
    names = onehot_names[cursor:cursor + len(categories)]
    feature_rows.append((field, "独热", len(categories), len(names), ", ".join(names)))
    cursor += len(categories)
feature_table = pd.DataFrame(feature_rows, columns=[
    "原始列名", "编码方式", "训练集类别数", "输出列数", "编码后的列名",
])
assert cursor == len(onehot_names)
assert feature_table["输出列数"].sum() == len(feature_names)
print("编码后维度：", len(feature_names))
print("训练与测试形状：", train_encoded.shape, test_encoded.shape)
print("各编码方式输出维度：", feature_table.groupby("编码方式")["输出列数"].sum().to_dict())
with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(feature_table)

encoded_train_hash = pd.util.hash_pandas_object(pd.DataFrame(train_encoded), index=False)
encoded_test_hash = pd.util.hash_pandas_object(pd.DataFrame(test_encoded), index=False)
encoded_overlap = int(encoded_test_hash.isin(set(encoded_train_hash)).sum())
print("编码后测试集共享输入指纹的行数：", encoded_overlap)

编码后维度： 91
训练与测试形状： (359977, 91) (89995, 91)
各编码方式输出维度： {'二值': 10, '数值': 14, '独热': 67}


,原始列名,编码方式,训练集类别数,输出列数,编码后的列名
0,duration,数值,无需统计,1,duration
1,src_bytes,数值,无需统计,1,src_bytes
2,dst_bytes,数值,无需统计,1,dst_bytes
3,missed_bytes,数值,无需统计,1,missed_bytes
4,src_pkts,数值,无需统计,1,src_pkts
5,src_ip_bytes,数值,无需统计,1,src_ip_bytes
6,dst_pkts,数值,无需统计,1,dst_pkts
7,dst_ip_bytes,数值,无需统计,1,dst_ip_bytes
8,dns_qclass,数值,无需统计,1,dns_qclass
9,dns_qtype,数值,无需统计,1,dns_qtype


编码后测试集共享输入指纹的行数： 54061


结论：普通划分根据训练集取值转换后得到 91 维，其中数值 14 维、二值 10 维、独热 67 维。表格列出全部原始列的输出数量与列名。转换前后均有 54,061 条测试记录与训练集共享输入指纹。

各数值列的单位与范围不同，PCA 又依据方差确定方向。为了减少数值尺度差异的影响，下一步对全部 91 列进行 Min-Max 处理。每列使用训练集的最小值和最大值，将训练值映射到 0—1 区间。测试集使用同一范围，超出范围的值单独报告，暂不裁剪。[Min-Max 定义](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html)说明了这一转换。DT、RF 通常不依赖这种缩放，不能预先将它写成检测效果提高的原因。

01 发现部分数值列存在长尾。为了检查主体记录是否被压缩到很小的区间，下面展示全部 91 列处理后的中位数、99 分位数及训练最大值。二值和独热列本来主要是 0/1，仍与其他输入一并展示。为了避免后续误用，同时保留并核对缩放前的矩阵。


In [13]:
# 8.根据训练集范围进行 Min-Max 缩放并检查长尾
from sklearn.preprocessing import MinMaxScaler

encoded_train_digest = hashlib.sha256(memoryview(np.ascontiguousarray(train_encoded))).hexdigest()
encoded_test_digest = hashlib.sha256(memoryview(np.ascontiguousarray(test_encoded))).hexdigest()
scaler = MinMaxScaler(copy=True)
train_scaled = scaler.fit_transform(train_encoded)
test_scaled = scaler.transform(test_encoded)
assert not np.shares_memory(train_scaled, train_encoded)
assert not np.shares_memory(test_scaled, test_encoded)
assert hashlib.sha256(memoryview(np.ascontiguousarray(train_encoded))).hexdigest() == encoded_train_digest
assert hashlib.sha256(memoryview(np.ascontiguousarray(test_encoded))).hexdigest() == encoded_test_digest

print("训练数据形状：", train_scaled.shape)
print("测试数据形状：", test_scaled.shape)
print("训练集非有限值数：", np.size(train_scaled) - np.isfinite(train_scaled).sum())
print("测试集非有限值数：", np.size(test_scaled) - np.isfinite(test_scaled).sum())
outside = np.argwhere((test_scaled < 0) | (test_scaled > 1))
print("测试集超出训练范围的矩阵值数：", len(outside))

# 汇总全部模型输入列的缩放结果
train_quantiles = np.quantile(train_scaled, [0.50, 0.99], axis=0)
test_quantiles = np.quantile(test_scaled, [0.50, 0.99], axis=0)

scaled_summary = pd.DataFrame({
    "训练中位数": train_quantiles[0],
    "训练99分位": train_quantiles[1],
    "训练最大值": train_scaled.max(axis=0),
    "测试中位数": test_quantiles[0],
    "测试99分位": test_quantiles[1],
}, index=feature_names)

scaled_summary.index.name = "模型输入列名"

print("展示的模型输入列数：", len(scaled_summary))

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
):
    display(scaled_summary)

outside_rows = []
for row_position, column_position in outside:
    outside_rows.append({
        "原行索引": int(X_test.index[row_position]),
        "特征名": feature_names[column_position],
        "原值": test_encoded[row_position, column_position],
        "训练最小值": scaler.data_min_[column_position],
        "训练最大值": scaler.data_max_[column_position],
        "Min-Max处理后的值": test_scaled[row_position, column_position],
    })
display(pd.DataFrame(outside_rows))

训练数据形状： (359977, 91)
测试数据形状： (89995, 91)
训练集非有限值数： 0
测试集非有限值数： 0
测试集超出训练范围的矩阵值数： 2
展示的模型输入列数： 91


,训练中位数,训练99分位,训练最大值,测试中位数,测试99分位
模型输入列名,,,,,
duration,5.569118e-10,7.821557e-04,1.0,5.354921e-10,8.121536e-04
src_bytes,0.000000e+00,7.751509e-07,1.0,0.000000e+00,7.751509e-07
dst_bytes,0.000000e+00,3.982265e-06,1.0,0.000000e+00,3.982265e-06
missed_bytes,0.000000e+00,0.000000e+00,1.0,0.000000e+00,0.000000e+00
src_pkts,3.965406e-06,7.137730e-05,1.0,3.965406e-06,7.137730e-05
src_ip_bytes,1.330033e-06,8.187598e-05,1.0,1.330033e-06,8.157535e-05
dst_pkts,0.000000e+00,7.834044e-05,1.0,0.000000e+00,8.356313e-05
dst_ip_bytes,0.000000e+00,1.870236e-04,1.0,0.000000e+00,1.870236e-04
dns_qclass,0.000000e+00,3.051665e-05,1.0,0.000000e+00,3.051665e-05


,原行索引,特征名,原值,训练最小值,训练最大值,Min-Max处理后的值
0,87989,dst_pkts,313943.00000,0.0,191472.00000,1.639629
1,440274,duration,93516.92917,0.0,93372.05713,1.001552


结论：普通划分的矩阵在缩放后没有改写缩放前的数值。训练与测试矩阵均没有非有限值。全部 91 列的分位数已展示，其中 `src_bytes` 的训练 99 分位数约为 0.000000775，显示大多数记录集中在区间低端。

测试集有 2 个值超过训练范围，分别来自 `dst_pkts` 和 `duration`，明细已全部列出。缩放限制不能单独推出模型效果。

前面的逐列核对已发现跨集合共享输入。为了避免同一处理后输入同时进入训练与测试，下一步按当前 38 列的精确取值分组，再将整组分到一侧。分组保留全部记录，不只取组内一条，也不根据标签拆开同组记录。这个条件只隔离当前输入组合，不证明设备、时间或网络事件也独立。

为了使测试数量尽量接近 20%，使用固定种子的五折分组候选，每次以一折作为测试部分。分组约束可能使行数偏离五分之一。为控制类别比例偏差，分折时考虑 `type` 分布，再将测试比例、攻击比例和类型分布与全表的偏差等权相加，选择总偏差最小的一折。五折仅用于生成候选划分，不是五折模型交叉验证，也不依据模型分数选折。


In [10]:
# 9.将相同输入归为一组，划分互不重叠的训练集和测试集
from sklearn.model_selection import StratifiedGroupKFold

group_ids = X.groupby(list(X.columns), sort=False, dropna=False).ngroup()
type_codes, type_names = pd.factorize(df["type"])
overall_type_share = np.bincount(type_codes, minlength=len(type_names)) / len(df)
overall_attack_share = y.mean()
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=random_seed)
best_split = None
best_score = float("inf")
for candidate_number, (train_pos, test_pos) in enumerate(splitter.split(X, df["type"], groups=group_ids)):
    test_share = len(test_pos) / len(X)
    attack_share = y.iloc[test_pos].mean()
    test_type_share = np.bincount(type_codes[test_pos], minlength=len(type_names)) / len(test_pos)
    type_distance = 0.5 * np.abs(test_type_share - overall_type_share).sum()
    score = 4 * abs(test_share - 0.20) + 4 * abs(attack_share - overall_attack_share) + 4 * type_distance
    if score < best_score:
        best_score = score
        best_split = (candidate_number, train_pos, test_pos, type_distance)

selected_candidate, group_train_pos, group_test_pos, selected_type_distance = best_split
X_group_train = X.iloc[group_train_pos].copy()
X_group_test = X.iloc[group_test_pos].copy()
y_group_train = y.iloc[group_train_pos].copy()
y_group_test = y.iloc[group_test_pos].copy()
type_group_train = df.loc[X_group_train.index, "type"]
type_group_test = df.loc[X_group_test.index, "type"]

print("候选折数与选中折：", 5, selected_candidate)
print("类别分布总变差：", selected_type_distance)
print("隔离划分训练行数：", len(X_group_train))
print("隔离划分测试行数：", len(X_group_test))
print("隔离划分实际测试比例：", len(X_group_test) / len(X))
print("隔离划分训练攻击比例：", y_group_train.mean())
print("隔离划分测试攻击比例：", y_group_test.mean())
print("两侧共享精确输入组合数：", len(set(group_ids.iloc[group_train_pos]) & set(group_ids.iloc[group_test_pos])))
display(pd.concat([
    type_group_train.value_counts().rename("训练集"),
    type_group_test.value_counts().rename("测试集"),
], axis=1).fillna(0).astype(int))
assert not set(group_ids.iloc[group_train_pos]) & set(group_ids.iloc[group_test_pos])

候选折数与选中折： 5 4
类别分布总变差： 0.011716961099172064
隔离划分训练行数： 360464
隔离划分测试行数： 89508
隔离划分实际测试比例： 0.19891904385161743
隔离划分训练攻击比例： 0.36044098717208933
隔离划分测试攻击比例： 0.3476449032488716
两侧共享精确输入组合数： 0


,训练集,测试集
type,,
normal,230538,58391
scanning,16723,3277
xss,16369,3631
dos,16000,4000
injection,16000,4000
ddos,16000,4000
password,16000,4000
ransomware,16000,4000
backdoor,16000,4000


结论：同一处理后的精确输入组合没有跨训练集和测试集。五个候选中，按预定的比例与分布偏差选择了编号为 4 的一折，编号从 0 开始。

隔离训练集为 360,464 行，测试集为 89,508 行。实际测试比例约为 19.89%，测试攻击比例约为 34.76%。训练与测试的类型数量已全部展示。分组避免跨集合共享同一输入，但不删除组内记录，也不解决同输入对应不同标签的问题。

普通划分只保留在本笔记用于处理核对，03 使用隔离划分。两次划分的训练记录不同，普通训练集确定的类别与范围不能直接沿用。所以下一步只依据隔离训练集重新完成独热编码和 Min-Max 处理，测试集使用这套规则。仅在测试集中出现的类别单独列出，便于解释转换结果。


In [11]:
# 10.根据隔离划分训练集重新完成独热编码和缩放
group_encoder = make_encoder()
group_train_encoded = group_encoder.fit_transform(X_group_train).astype(np.float64, copy=False)
group_test_encoded = group_encoder.transform(X_group_test).astype(np.float64, copy=False)
group_feature_names = group_encoder.get_feature_names_out()
group_scaler = MinMaxScaler(copy=True)
group_train_scaled = group_scaler.fit_transform(group_train_encoded)
group_test_scaled = group_scaler.transform(group_test_encoded)
assert not np.shares_memory(group_train_scaled, group_train_encoded)
assert not np.shares_memory(group_test_scaled, group_test_encoded)
assert np.isfinite(group_train_scaled).all() and np.isfinite(group_test_scaled).all()
print("隔离划分编码维度：", len(group_feature_names))
print("隔离划分训练与测试形状：", group_train_scaled.shape, group_test_scaled.shape)
print("普通分层划分存在而隔离划分训练侧未形成的编码列：", sorted(set(feature_names) - set(group_feature_names)))
print("隔离划分独有的编码列：", sorted(set(group_feature_names) - set(feature_names)))
unseen_categories = []
for field in category_cols:
    unseen = set(X_group_test[field].unique()) - set(X_group_train[field].unique())
    for value in sorted(unseen):
        unseen_categories.append({
            "原始列名": field,
            "仅测试侧出现的取值": value,
            "测试行数": int(X_group_test[field].eq(value).sum()),
        })
display(pd.DataFrame(unseen_categories))
print("隔离划分测试集超出训练范围的矩阵值数：", int(((group_test_scaled < 0) | (group_test_scaled > 1)).sum()))
group_train_hash = pd.util.hash_pandas_object(pd.DataFrame(group_train_encoded), index=False)
group_test_hash = pd.util.hash_pandas_object(pd.DataFrame(group_test_encoded), index=False)
print("隔离划分编码后共享输入指纹的测试行数：", int(group_test_hash.isin(set(group_train_hash)).sum()))

隔离划分编码维度： 88
隔离划分训练与测试形状： (360464, 88) (89508, 88)
普通分层划分存在而隔离划分训练侧未形成的编码列： ['http_trans_depth_3', 'http_trans_depth_4', 'http_trans_depth_5']
隔离划分独有的编码列： []


,原始列名,仅测试侧出现的取值,测试行数
0,http_trans_depth,3,1
1,http_trans_depth,4,1
2,http_trans_depth,5,1


隔离划分测试集超出训练范围的矩阵值数： 5
隔离划分编码后共享输入指纹的测试行数： 0


结论：隔离划分得到 88 维，训练形状为 `(360464, 88)`，测试形状为 `(89508, 88)`。它比普通划分少 3 列，全部来自 `http_trans_depth` 的 `3`、`4`、`5`。这三个取值各在隔离测试集中出现一次，在隔离训练集中未出现，所以没有生成对应列。记录仍保留，该原始列的独热部分全部为 0。

隔离划分没有独有的编码列，空列表 `[]` 表示数量为 0。未见类别表只列训练集中没有的取值，其他类别不在这张筛查表中。不同集合可以共有协议、服务等单列类别，隔离条件针对整行输入组合。

处理后的矩阵中也没有跨集合共享输入指纹。隔离测试集有 5 个矩阵值超出训练范围，未作裁剪；它们按单元格计数，不等于 5 条记录。

03 需要独立读取已处理输入。所以下一步保存隔离划分的矩阵、标签和对应信息，并重新读取核对。


In [ ]:
# 11.保存隔离划分的模型输入并重新读取核对
import os
import tempfile

processed_dir = root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
output_path = processed_dir / "model_inputs.npz"
payload = {
    "group_X_train": group_train_scaled,
    "group_X_test": group_test_scaled,
    "group_y_train": y_group_train.to_numpy(dtype=np.int8),
    "group_y_test": y_group_test.to_numpy(dtype=np.int8),
    "group_train_index": X_group_train.index.to_numpy(dtype=np.int64),
    "group_test_index": X_group_test.index.to_numpy(dtype=np.int64),
    "group_train_type": type_group_train.to_numpy(dtype=str),
    "group_test_type": type_group_test.to_numpy(dtype=str),
    "group_feature_names": np.asarray(group_feature_names, dtype=str),
    "input_fields": np.asarray(X.columns, dtype=str),
    "raw_sha256": np.asarray(raw_sha256),
    "random_seed": np.asarray(random_seed, dtype=np.int64),
    "schema_version": np.asarray("1"),
}

needs_save = not output_path.exists()
if output_path.exists():
    previous_split_keys = {
        "X_train", "X_test", "y_train", "y_test", "train_index",
        "test_index", "train_type", "test_type", "feature_names",
    }
    with np.load(output_path, allow_pickle=False) as stored:
        stored_keys = set(stored.files)
        assert stored_keys in (set(payload), set(payload) | previous_split_keys), "现有文件的数据项与预期不同"
        for key, value in payload.items():
            assert np.array_equal(stored[key], value), f"现有隔离划分内容与当前运行不同：{key}"
        needs_save = stored_keys != set(payload)

if needs_save:
    temporary_path = None
    try:
        with tempfile.NamedTemporaryFile(dir=processed_dir, suffix=".npz", delete=False) as temporary:
            temporary_path = Path(temporary.name)
            np.savez_compressed(temporary, **payload)
        os.replace(temporary_path, output_path)
    finally:
        if temporary_path is not None and temporary_path.exists():
            temporary_path.unlink()

with np.load(output_path, allow_pickle=False) as stored:
    assert set(stored.files) == set(payload), "保存文件含有多余或缺少的数据项"
    for key, value in payload.items():
        assert np.array_equal(stored[key], value), f"重新读取内容与当前运行不同：{key}"
    assert stored["group_X_train"].shape == (len(y_group_train), len(group_feature_names))
    assert stored["group_X_test"].shape == (len(y_group_test), len(group_feature_names))
    assert stored["raw_sha256"].item() == raw_sha256
    assert stored["group_feature_names"].tolist() == group_feature_names.tolist()
    print("隔离划分矩阵：", stored["group_X_train"].shape, stored["group_X_test"].shape)
    print("隔离划分特征名数：", len(stored["group_feature_names"]))

隔离划分矩阵： (360464, 88) (89508, 88)
隔离划分特征名数： 88


结论：隔离划分的输入已保存到 `data/processed/model_inputs.npz`。重新读取后，各数据项与本次运行一致。文件只含隔离划分与对应设置，不含普通划分的模型矩阵。

保存内容包括矩阵、标签、原行索引、流量类型和特征名，也记录原始输入列名、CSV 哈希和随机种子。主要数据项如下。

| 数据项 | 含义 | 数量 |
| --- | --- | --- |
| `group_X_train` | 隔离训练集输入，供模型学习 | 360,464 行 × 88 列 |
| `group_X_test` | 隔离测试集输入，供模型预测 | 89,508 行 × 88 列 |
| `group_y_train` | 各训练记录对应的标签 | 360,464 个 |
| `group_y_test` | 各测试记录对应的标签，用于核对预测 | 89,508 个 |
| `group_feature_names` | 按矩阵列顺序排列的输入列名 | 88 个 |

输入准备完成还不能判断检测效果。所以下一步由 03 读取该文件，核对输入与标签是否对应，再开始比较。
